In [3]:
#starter cell
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/AI4_Data')
RAW = ROOT / 'Raw'
PROC = ROOT / 'Processed'
BB = ROOT / 'ByteEntropy_Temp'  #new folder for byteentropy batches

RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
BB.mkdir(parents=True, exist_ok=True)

print("Folders ready ✔️")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folders ready ✔️


In [4]:
import json

byte_raw = []

with open(RAW / "train_features_1.jsonl", "r") as f:
    for line in f:
        obj = json.loads(line)
        byte_raw.append(obj["byteentropy"])  #pull only the needed column

len(byte_raw)

158158

In [6]:
#flatten the column in batches
import numpy as np
import pandas as pd

batch_size = 5000

for batch_id, start in enumerate(range(0, len(byte_raw), batch_size)):
    end = min(start + batch_size, len(byte_raw)) #min: doesn't go out of bounds

    #slice raw lists (each item should be length-256 list)
    batch_slice = byte_raw[start:end]

    #build directly as a DataFrame
    batch_df = pd.DataFrame(batch_slice)

    #ensure exactly 256 columns (0..255); fill missing with NaN
    #(handles any odd rows that aren't length-256)
    batch_df = batch_df.reindex(columns=range(256), fill_value=np.nan)

    #name columns with the final schema we want
    batch_df.columns = [f'byte.{i}' for i in range(256)]

    #clean dtypes: None → NaN → 0, then cast to float32 for smaller files
    batch_df = batch_df.replace({None: np.nan}).astype('float32').fillna(0.0)

    #write immediately to keep RAM low
    out_path = BB / f"byte_batch_{batch_id}.parquet"
    batch_df.to_parquet(out_path, index=False)

    #progress + free memory
    print(f"Saved batch {batch_id} ({start} to {end})")
    del batch_slice, batch_df

Saved batch 0 (0 to 5000)
Saved batch 1 (5000 to 10000)
Saved batch 2 (10000 to 15000)
Saved batch 3 (15000 to 20000)
Saved batch 4 (20000 to 25000)
Saved batch 5 (25000 to 30000)
Saved batch 6 (30000 to 35000)
Saved batch 7 (35000 to 40000)
Saved batch 8 (40000 to 45000)
Saved batch 9 (45000 to 50000)
Saved batch 10 (50000 to 55000)
Saved batch 11 (55000 to 60000)
Saved batch 12 (60000 to 65000)
Saved batch 13 (65000 to 70000)
Saved batch 14 (70000 to 75000)
Saved batch 15 (75000 to 80000)
Saved batch 16 (80000 to 85000)
Saved batch 17 (85000 to 90000)
Saved batch 18 (90000 to 95000)
Saved batch 19 (95000 to 100000)
Saved batch 20 (100000 to 105000)
Saved batch 21 (105000 to 110000)
Saved batch 22 (110000 to 115000)
Saved batch 23 (115000 to 120000)
Saved batch 24 (120000 to 125000)
Saved batch 25 (125000 to 130000)
Saved batch 26 (130000 to 135000)
Saved batch 27 (135000 to 140000)
Saved batch 28 (140000 to 145000)
Saved batch 29 (145000 to 150000)
Saved batch 30 (150000 to 155000)
S

In [7]:
#combine all saved batches
batch_files = sorted(BB.glob("byte_batch_*.parquet"))

byte_list = [pd.read_parquet(f) for f in batch_files]
byte_tf1 = pd.concat(byte_list, axis=0, ignore_index=True)
print("byte_tf1 shape:", byte_tf1.shape)
byte_tf1.head()

byte_tf1 shape: (158158, 256)


,byte.0,byte.1,byte.2,byte.3,byte.4,byte.5,byte.6,byte.7,byte.8,byte.9,...,byte.246,byte.247,byte.248,byte.249,byte.250,byte.251,byte.252,byte.253,byte.254,byte.255
0,24434.0,11.0,18.0,5.0,9.0,8.0,35.0,21.0,3.0,4.0,...,16462.0,15966.0,14060.0,13914.0,13731.0,13957.0,14147.0,13756.0,13708.0,13644.0
1,2006.0,2.0,3.0,0.0,4.0,3.0,3.0,1.0,9.0,1.0,...,371.0,332.0,236.0,212.0,307.0,319.0,347.0,173.0,573.0,330.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,329804.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,62623.0,62680.0,62537.0,62711.0,62143.0,62652.0,62606.0,62535.0,62239.0,62039.0


In [8]:
#save this
byte_tf1.to_parquet(PROC / "tf1_ByteEntropy.parquet", index=False)
print("saved tf1_ByteEntropy.parquet ✔️")

saved tf1_ByteEntropy.parquet ✔️
